<a href="https://colab.research.google.com/github/Text-Machine/temporal-adapters/blob/train/temporal-sft-train.ipynb" target="_parent\"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

# Notebook for data preperation

In [2]:
from datasets import Dataset, load_dataset

In [3]:
dataset = load_dataset("Kaspar/key_phrases_dataset")

README.md:   0%|          | 0.00/453 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

model_name = "meta-llama/Llama-2-7b-hf"
# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)

# Format prompt-completion pairs
def formatting_func(example):
    return f"### Prompt:\n{example['prompt']}\n\n### Response:\n{example['completion']}"

# Training configuration
training_args = TrainingArguments(
    output_dir="./sft-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=500,
    fp16=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    tokenizer=tokenizer,
    formatting_func=formatting_func,
    args=training_args,
    max_seq_length=1024,
)

trainer.train()

trainer.save_model("./sft-model")